In [3]:
import numpy as np
import pandas as pd

import zarr
import tifffile

from pathlib import Path

import gseapy as gp
import scanpy
import novae

import loki

In [2]:
gene_list_dir = "/nfs/team361/dj16/data/MSigDB/"
path_to_gmt = Path(gene_list_dir, "HSIAO_HOUSEKEEPING_GENES.v2025.1.Hs.gmt")
house_keeping_genes_gmt = gp.parser.read_gmt(str(path_to_gmt))

df_house_keeping_genes = pd.DataFrame(house_keeping_genes_gmt).rename(columns={'HSIAO_HOUSEKEEPING_GENES':'genesymbol'})

df_house_keeping_genes.to_csv(Path(gene_list_dir, "housekeeping_genes.csv"))

In [15]:
# tissue_type = "lung_cancer"
# data_dir = "/nfs/team361/dj16/data/Xenium_Prime_Human_Lung_Cancer_FFPE/"
# image_file = Path(data_dir, "Xenium_Prime_Human_Lung_Cancer_FFPE_he_image.ome.tif")
# transform_for_alignment = np.loadtxt(Path(data_dir, "Xenium_Prime_Human_Lung_Cancer_FFPE_he_imagealignment.csv"), delimiter=",")
# spatial_data = scanpy.read_10x_h5(Path(data_dir, "cell_feature_matrix.h5"))
# cells_df = pd.read_csv(Path(data_dir,"cells.csv.gz"))


# tissue_type = "breast_cancer"
# data_dir = "/nfs/team361/dj16/data/Xenium_Prime_Human_Breast_Cancer_FFPE//"
# image_file = Path(data_dir, "Xenium_Prime_Breast_Cancer_FFPE_he_image.ome.tif")
# transform_for_alignment = np.loadtxt(Path(data_dir, "Xenium_Prime_Breast_Cancer_FFPE_he_imagealignment.csv"), delimiter=",")
# spatial_data = scanpy.read_10x_h5(Path(data_dir, "cell_feature_matrix.h5"))
# cells_df = pd.read_csv(Path(data_dir, "cells.csv.gz"))


tissue_type = "ovarian_cancer"
data_dir = "/nfs/team361/dj16/data/Xenium_Prime_Ovarian_Cancer_FFPE_XRrun/"
image_file = Path(data_dir, "Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_he_image.ome.tif")
transform_for_alignment = np.loadtxt(Path(data_dir, "Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_he_imagealignment.csv"), delimiter=",")
spatial_data = scanpy.read_10x_h5(Path(data_dir, "cell_feature_matrix.h5"))
cells_df = pd.read_csv(Path(data_dir, "cells.csv.gz"))
celltype_file = Path(data_dir, "Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_cell_groups.csv")


# tissue_type = "lymph_node"
# data_dir = "/nfs/team361/dj16/data/Xenium_Prime_Human_Lymph_Node_Reactive_FFPE/"
# image_file = Path(data_dir, "Xenium_Prime_Human_Lymph_Node_Reactive_FFPE_he_image.ome.tif")
# transform_for_alignment = np.loadtxt(Path(data_dir, "Xenium_Prime_Human_Lymph_Node_Reactive_FFPE_he_imagealignment.csv"), delimiter=",")
# spatial_data = scanpy.read_10x_h5(Path(data_dir, "cell_feature_matrix.h5"))
# cells_df = pd.read_csv(Path(data_dir, "cells.csv.gz"))
# celltype_file = Path(data_dir, "Xenium_Prime_Human_Lymph_Node_Reactive_FFPE_cell_types.csv")

# Align H&E image

In [4]:
# ===== Load image data ===== 
image_data = tifffile.imread(image_file)

# ref: https://kb.10xgenomics.com/hc/en-us/articles/35386990499853-How-can-I-convert-coordinates-between-H-E-image-and-Xenium-data

transform_inv = np.linalg.inv(transform_for_alignment)

# ===== Load spatial data ===== 
# Extract x, y
x = cells_df["x_centroid"] / 0.2125
y = cells_df["y_centroid"] / 0.2125

# Make homogeneous coordinates: shape (3, N)
xy_hom = np.vstack([x.to_numpy(), y.to_numpy(), np.ones_like(x)])

# Apply affine transformation
xy_aligned = transform_inv @ xy_hom

# Save results back to DataFrame
cells_df["x_pixel"] = xy_aligned[0]
cells_df["y_pixel"] = xy_aligned[1]

cells_df.set_index("cell_id", inplace=True)

spatial_data.obs = cells_df.loc[spatial_data.obs_names].copy()

spatial_data.obsm["spatial"] = spatial_data.obs[["x_pixel", "y_pixel"]].to_numpy()

# Add novae embedding

In [5]:
x1 = spatial_data

In [6]:
# Spatial encoder
spatial_encoder = novae.Novae.from_pretrained("MICS-Lab/novae-human-0")

# spatial embedding
novae.spatial_neighbors(x1, radius=80)
spatial_encoder.compute_representations(
    x1, zero_shot=True
)  # x should be adata

[INFO] (novae.utils._build) Computing graph on 407,124 cells (coord_type=generic, delaunay=True, radius=[0.0, 80.0], n_neighs=None)
/nfs/team361/dj16/pypoetry/virtualenvs/model-_goj29qN-py3.12/lib/python3.12/site-packages/legacy_api_wrap/__init__.py:82: UserWarning: Some cells have zero counts
  return fn(*args_all, **kw)
[INFO] (novae.utils._validate) Preprocessed 1 adata object(s) with sc.pp.normalize_total and sc.pp.log1p (raw counts were saved in adata.layers['counts'])


Computing representations:   0%|          | 0/784 [00:00<?, ?it/s]

# Add NEMO embedding
- checkpoint: /nfs/team361/sb75/nichejepa-reproducibility/artifacts/hst_corpus_80m/11062025_183320_055/nichejepa-ep10_0.pth.tar

In [7]:
# TODO

# Add cell type compostion

In [8]:
if tissue_type in ["ovarian_cancer", "lymph_node"]:
    cell_types = pd.read_csv(celltype_file, index_col=0, names=['cell_types'])
    x1.obs = x1.obs.join(cell_types)

In [9]:
x1.obsp

PairwiseArrays with keys: spatial_connectivities, spatial_distances, spatial_distances_local, spatial_distances_pair

# Save anndata

In [10]:
scanpy.write(Path(data_dir, f"spatial_preprocessed_{tissue_type}.h5"), x1)

# Harmonize cell type annotations

In [1]:
manual_map = {
    "Macrophages": "Macrophage",
    "T and NK Cells": "T Cell",  # 또는 "NK T Cell"
    "Stromal Associated Fibroblasts": "Stromal Cell",
    "Stromal Associated Endothelial Cells": "Endothelial Cell",
    "Smooth Muscle Cells": "Smooth Muscle Cell",  # a에 없으면 NO_MATCH
    "Pericytes": "Pericyte",  # a에 없음
    "Granulosa Cells": "NO_MATCH",  # 완전히 다름
    "Unassigned": "NO_MATCH",
    # Tumor 계열은 대부분 NO_MATCH 처리
}

In [4]:
data_dir = "/nfs/team361/dj16/data/Xenium_Prime_Ovarian_Cancer_FFPE_XRrun/"
tmp = scanpy.read_h5ad(Path(data_dir, f"spatial_preprocessed_ovarian_cancer.h5"))
tmp.obs['cell_types'] = tmp.obs['cell_types'].map(lambda x:manual_map[x] if x in manual_map else x)
a = tmp.obs["cell_types"].unique()#.dropna()

In [5]:
data_dir = "/nfs/team361/dj16/data/Xenium_Prime_Human_Lymph_Node_Reactive_FFPE/"
tmp = scanpy.read_h5ad(Path(data_dir, f"spatial_preprocessed_lymph_node.h5"))
tmp.obs['cell_types'] = tmp.obs['cell_types'].map(lambda x:manual_map[x] if x in manual_map else x)
b = tmp.obs["cell_types"].unique().dropna()

In [6]:
set(a).intersection(set(b))


{'Endothelial Cell', 'Macrophage', 'Stromal Cell', 'T Cell'}

In [7]:
for item in a:
    print(item)

Proliferative Tumor Cells
Tumor Cells
Ciliated Epithelial Cells
Inflammatory Tumor Cells
Macrophage
T Cell
SOX2-OT+ Tumor Cells
Tumor Associated Fibroblasts
VEGFA+ Tumor Cells
Malignant Cells Lining Cyst
Tumor Associated Endothelial Cells
Fallopian Tube Epithelium
NO_MATCH
Stromal Cell
Endothelial Cell
Smooth Muscle Cell
nan
Pericyte


In [8]:
for item in b:
    print(item)

Endothelial Cell
CD4+ αβ Memory T Cell
Hematopoietic Stem Cell
Innate Lymphoid Cell
Classical Monocyte
Regulatory T Cell
CD1c+ Myeloid Dendritic Cell
Memory B Cell
B Cell
CD8+ αβ Memory T Cell
Macrophage
Naive Thymus Derived CD4+ αβ T Cell
Intermediate Monocyte
Naive B Cell
Stromal Cell
Effector CD4+ αβ T Cell
Effector CD8+ αβ T Cell
Plasma Cell
Plasmacytoid Dendritic Cell
T Cell
CD141+ Myeloid Dendritic Cell
Mature NK T Cell
Type I NK T Cell
Non Classical Monocyte
Mast Cell
Mature Conventional Dendritic Cell
Erythrocyte
Neutrophil


# Tiff to zarr

In [ ]:
## image processing
image = tifffile.imread(image_file)
z = zarr.open(Path(data_dir,'image.zarr'), mode='w', shape=image.shape, chunks=(256, 256, 3), dtype='uint8')
z[:] = image
## indices processing
# 이미지 크기
H, W, C = image.shape
# 패치 크기 및 스트라이드
patch_size = 256
stride = 256
# 시작점 계산 (top-left corner)
y_coords = np.arange(0, H - patch_size + 1, stride)
x_coords = np.arange(0, W - patch_size + 1, stride)

# 모든 (y, x) 시작점 조합 생성
patch_indices = np.array([(y, x) for y in y_coords for x in x_coords], dtype=np.int32)  # shape (N, 2)
patch_indices = patch_indices[:, :, np.newaxis] # shape: (N, 2, 1)

# Zarr로 저장
zarr.save(Path(data_dir, 'indices.zarr'), patch_indices)